# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [1]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Weselmy się!
2. Andrzej Dudziński. Przypadek artysty niemożliwego
3. Piąty element – MIŁOŚĆ. Wystawa Krakowskiego Klubu Fotograficznego
4. Planszówki w bibliotece
5. Godność osoby ludzkiej
6. Gotyk w Karpatach
7. Wspólnie. 20 lat kolektywu Sputnik Photos
8. Skarby z Łowicza
9. Kuchnia od kuchni
10. Suknia (jednak) zdobi człowieka!

Czas wykonania: 2.45s


In [2]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Weselmy się!
2. Andrzej Dudziński. Przypadek artysty niemożliwego
3. Piąty element – MIŁOŚĆ. Wystawa Krakowskiego Klubu Fotograficznego
4. Planszówki w bibliotece
5. Godność osoby ludzkiej
6. Gotyk w Karpatach
7. Wspólnie. 20 lat kolektywu Sputnik Photos
8. Skarby z Łowicza
9. Kuchnia od kuchni
10. Suknia (jednak) zdobi człowieka!

Czas wykonania (wątki): 0.63s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [3]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [4]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 10 procesach (rdzeniach)...
Zakończono w czasie 0.32s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [9]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

def fetch_fact(_):
    """Pobiera pojedynczy fakt. Podkreślnik '_' oznacza, że ignorujemy argument (numer iteracji)."""
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    try:
        response = requests.get(CAT_API_URL, headers=headers, timeout=5)
        response.raise_for_status()
        return response.json().get('fact', 'Brak faktu')
    except Exception:
        return "Błąd pobierania danych z API"

print("Rozpoczynam pobieranie sekwencyjne 20 faktów...")
start_seq = time.time()

seq_results = []
for i in range(20):
    seq_results.append(fetch_fact(i))

time_seq = time.time() - start_seq
print(f"Zakończono sekwencyjnie w: {time_seq:.2f}s\n")

print("Rozpoczynam pobieranie wielowątkowe 20 faktów...")
start_thread = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    thread_results = list(executor.map(fetch_fact, range(20)))

time_thread = time.time() - start_thread
print(f"Zakończono wielowątkowo w: {time_thread:.2f}s\n")

print("=== PODSUMOWANIE ===")
print(f"Przyspieszenie: {time_seq / time_thread:.2f}x szybciej")
print(f"Przykładowy z pobranych faktów: {thread_results[0]}")

Rozpoczynam pobieranie sekwencyjne 20 faktów...
Zakończono sekwencyjnie w: 3.72s

Rozpoczynam pobieranie wielowątkowe 20 faktów...
Zakończono wielowątkowo w: 0.55s

=== PODSUMOWANIE ===
Przyspieszenie: 6.81x szybciej
Przykładowy z pobranych faktów: Cats' eyes shine in the dark because of the tapetum, a reflective layer in the eye, which acts like a mirror.


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [11]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time

q = queue.Queue()
stop_signal = object()

def producer():
    for i in range(1, 21):
        q.put(i)
        time.sleep(0.005)
    q.put(stop_signal)
    q.put(stop_signal)

def even_consumer():
    while True:
        item = q.get()
        if item is stop_signal:
            q.task_done()
            break
        if item % 2 == 0:
            print(f"[PARZYSTE] Liczba: {item} (Postęp: {item}/20)")
            q.task_done()
        else:
            q.put(item)
            q.task_done()
            time.sleep(0.001)

def odd_consumer():
    while True:
        item = q.get()
        if item is stop_signal:
            q.task_done()
            break
        if item % 2 != 0:
            print(f"[NIEPARZYSTE] Liczba: {item} (Postęp: {item}/20)")
            q.task_done()
        else:
            q.put(item)
            q.task_done()
            time.sleep(0.001)

t1 = threading.Thread(target=producer)
t2 = threading.Thread(target=even_consumer)
t3 = threading.Thread(target=odd_consumer)

t1.start()
t2.start()
t3.start()

q.join()
t1.join()
t2.join()
t3.join()

print("Zrobione!")

[NIEPARZYSTE] Liczba: 1 (Postęp: 1/20)
[PARZYSTE] Liczba: 2 (Postęp: 2/20)
[NIEPARZYSTE] Liczba: 3 (Postęp: 3/20)
[PARZYSTE] Liczba: 4 (Postęp: 4/20)
[NIEPARZYSTE] Liczba: 5 (Postęp: 5/20)
[PARZYSTE] Liczba: 6 (Postęp: 6/20)
[NIEPARZYSTE] Liczba: 7 (Postęp: 7/20)
[PARZYSTE] Liczba: 8 (Postęp: 8/20)
[NIEPARZYSTE] Liczba: 9 (Postęp: 9/20)
[PARZYSTE] Liczba: 10 (Postęp: 10/20)
[NIEPARZYSTE] Liczba: 11 (Postęp: 11/20)
[PARZYSTE] Liczba: 12 (Postęp: 12/20)
[NIEPARZYSTE] Liczba: 13 (Postęp: 13/20)
[PARZYSTE] Liczba: 14 (Postęp: 14/20)
[NIEPARZYSTE] Liczba: 15 (Postęp: 15/20)
[PARZYSTE] Liczba: 16 (Postęp: 16/20)
[NIEPARZYSTE] Liczba: 17 (Postęp: 17/20)
[PARZYSTE] Liczba: 18 (Postęp: 18/20)
[NIEPARZYSTE] Liczba: 19 (Postęp: 19/20)
[PARZYSTE] Liczba: 20 (Postęp: 20/20)
Zrobione!


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [12]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    liczby = list(range(1, 10001))
    rdzenie = multiprocessing.cpu_count()
    
    start_time = time.time()
    
    with multiprocessing.Pool(processes=rdzenie) as pool:
        wyniki = pool.map(calculate_power_sum, liczby, chunksize=100)
        
    czas_wykonania = time.time() - start_time
    
    print(f"Przetworzono {len(liczby)} liczb na {rdzenie} rdzeniach.")
    print(f"Czas wieloprocesowy: {czas_wykonania:.4f} sekund.")

Przetworzono 10000 liczb na 10 rdzeniach.
Czas wieloprocesowy: 0.1648 sekund.
